In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2.1 Labeling Data and Creating Directory Structure

In this step, we define what constitutes a 'malignant' diagnosis based on the `dx` column in the metadata. A new `label` column is created, assigning `1` for malignant and `0` for benign. We also set up the main `DATASET_DIR` and create subdirectories for `train`, `val`, and `test` splits, each containing `benign` and `malignant` subfolders. This hierarchical structure is essential for later use with `ImageDataGenerator`.

In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/skin_cancer_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

DATA_DIR = f"{PROJECT_DIR}/data"
os.makedirs(DATA_DIR, exist_ok=True)
%cd $DATA_DIR

/content/drive/MyDrive/skin_cancer_project/data


**Downloading dataset from HAM1000 collected by Harvard**

In [6]:
!wget https://dataverse.harvard.edu/api/access/datafile/3172585 -O images_part1.zip
!wget https://dataverse.harvard.edu/api/access/datafile/3172584 -O images_part2.zip
!wget https://dataverse.harvard.edu/api/access/datafile/3172586 -O metadata.csv

KeyboardInterrupt: 

In [5]:
!unzip images_part1.zip
!unzip images_part2.zip

Streaming output truncated to the last 5000 lines.
  inflating: ISIC_0029321.jpg        
  inflating: ISIC_0029322.jpg        
  inflating: ISIC_0029323.jpg        
  inflating: ISIC_0029324.jpg        
  inflating: ISIC_0029325.jpg        
  inflating: ISIC_0029326.jpg        
  inflating: ISIC_0029327.jpg        
  inflating: ISIC_0029328.jpg        
  inflating: ISIC_0029329.jpg        
  inflating: ISIC_0029330.jpg        
  inflating: ISIC_0029331.jpg        
  inflating: ISIC_0029332.jpg        
  inflating: ISIC_0029333.jpg        
  inflating: ISIC_0029334.jpg        
  inflating: ISIC_0029335.jpg        
  inflating: ISIC_0029336.jpg        
  inflating: ISIC_0029337.jpg        
  inflating: ISIC_0029338.jpg        
  inflating: ISIC_0029339.jpg        
  inflating: ISIC_0029340.jpg        
  inflating: ISIC_0029341.jpg        
  inflating: ISIC_0029342.jpg        
  inflating: ISIC_0029343.jpg        
  inflating: ISIC_0029344.jpg        
  inflating: ISIC_0029345.jpg        

In [6]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/skin_cancer_project/HAM10000_metadata.csv")
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [7]:
malignant = ['mel', 'bcc', 'akiec']

df['label'] = df['dx'].apply(lambda x: 1 if x in malignant else 0)
df['label'].value_counts()

DATASET_DIR = f"{PROJECT_DIR}/dataset"

for split in ['train', 'val', 'test']:
    for cls in ['benign', 'malignant']:
        os.makedirs(f"{DATASET_DIR}/{split}/{cls}", exist_ok=True)

In [8]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

In [13]:
import shutil
IMAGE_DIR = DATA_DIR # Corrected: Images are directly in DATA_DIR
def find_image(img_id):
    path = f"{IMAGE_DIR}/{img_id}.jpg"
    if os.path.exists(path):
        return path
    return None

def copy_images(df_split, split_name):
    for _, row in df_split.iterrows():
        src = find_image(row['image_id'])
        if src:
            label = "malignant" if row['label'] == 1 else "benign"
            dst = f"{DATASET_DIR}/{split_name}/{label}/{row['image_id']}.jpg"
            shutil.copy(src, dst)

copy_images(train_df, "train")
copy_images(val_df, "val")
copy_images(test_df, "test")

Now all data are downloaded and re-labeled in a binary way (malignant or benign) and well balanced for testing, training, evaluation.

The next Step is starting to fine tune the inception model.

In [1]:
import tensorflow as tf
import os

Image generator and augmentation

In [7]:
IMG_SIZE = 299
BATCH_SIZE = 32

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1.0,
    rotation_range=15,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1]
)

val_test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./127.5,
    preprocessing_function=lambda x: x - 1.0
)


BASE_DATASET = "/content/drive/MyDrive/skin_cancer_project/dataset"

train_data = train_gen.flow_from_directory(
    f"{BASE_DATASET}/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_test_gen.flow_from_directory(
    f"{BASE_DATASET}/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = val_test_gen.flow_from_directory(
    f"{BASE_DATASET}/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)


Found 7010 images belonging to 2 classes.
Found 1502 images belonging to 2 classes.
Found 1503 images belonging to 2 classes.


Loading the pretrained model

In [8]:
base_model = tf.keras.applications.InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(base_model.input, output)


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [11]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 3500s 16s/step - accuracy: 0.7824 - auc: 0.6728 - loss: 0.5185 - precision: 0.3880 - recall: 0.1379 - val_accuracy: 0.8189 - val_auc: 0.8172 - val_loss: 0.3841 - val_precision: 0.6180 - val_recall: 0.1877
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 254s 1s/step - accuracy: 0.8247 - auc: 0.8069 - loss: 0.3855 - precision: 0.5877 - recall: 0.2526 - val_accuracy: 0.8016 - val_auc: 0.8340 - val_loss: 0.3997 - val_precision: 0.4920 - val_recall: 0.5222
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 249s 1s/step - accuracy: 0.8237 - auc: 0.8172 - loss: 0.3837 - precision: 0.6082 - recall: 0.2747 - val_accuracy: 0.8209 - val_auc: 0.8386 - val_loss: 0.3683 - val_precision: 0.6818 - val_recall: 0.1536
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 247s 1s/step - accuracy: 0.8262 - auc: 0.8272 - loss: 0.3772 - precision: 0.6469 - recall: 0.2549 - val_accuracy: 0.8202 - val_auc: 0.8434 - val_loss: 0.3648 - val_precision: 0.6456 - val_recall: 0.1741
Epoch 5/15
220/220 ━━━━━━━

The the head classifier had been finished, now Unfreezing the higher layers for the fine tuning

In [12]:
for layer in base_model.layers[-80:]:
    layer.trainable = True

Recompiling the model after changing the trainable layers

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

Now Fine-tuning training

In [15]:
history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.8079 - auc: 0.8149 - loss: 0.4050 - precision: 0.4966 - recall: 0.4312 - val_accuracy: 0.8356 - val_auc: 0.8456 - val_loss: 0.3661 - val_precision: 0.6474 - val_recall: 0.3447
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 243s 1s/step - accuracy: 0.8297 - auc: 0.8583 - loss: 0.3553 - precision: 0.6119 - recall: 0.4361 - val_accuracy: 0.8429 - val_auc: 0.8599 - val_loss: 0.3533 - val_precision: 0.6939 - val_recall: 0.3481
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 242s 1s/step - accuracy: 0.8453 - auc: 0.8845 - loss: 0.3201 - precision: 0.6588 - recall: 0.4199 - val_accuracy: 0.8522 - val_auc: 0.8666 - val_loss: 0.3460 - val_precision: 0.7006 - val_recall: 0.4232
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 244s 1s/step - accuracy: 0.8664 - auc: 0.9069 - loss: 0.2907 - precision: 0.7292 - recall: 0.4592 - val_accuracy: 0.8515 - val_auc: 0.8756 - val_loss: 0.3361 - val_precision: 0.6446 - val_recall: 0.5324
Epoch 5/10
220/220 ━━━━━━━━━

After finishing the fine-tuning, now we save the model and extract the .tflite for the flutter app.

In [18]:
# Save the full model (architecture + weights + optimizer)
model.save("skin_cancer_model.h5")
model.save_weights("model_weights.weights.h5")

In [22]:
import tensorflow as tf
import os # Import os module to use MODELS_DIR

# Load the Keras model (if restarting Colab)
model = tf.keras.models.load_model(os.path.join(MODELS_DIR, "skin_cancer_model.h5"))

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Optional optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization for mobile

tflite_model = converter.convert()

# Save TFLite file
with open(os.path.join(MODELS_DIR, "skin_cancer_model.tflite"), "wb") as f: # Save TFLite to MODELS_DIR as well
    f.write(tflite_model)

print("TFLite model saved!")

Saved artifact at '/tmp/tmp653961ar'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  133319889850192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889850384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889847888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889846736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889851728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889846544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889851536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889849424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889852496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889853648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133319889849232

# Skin Cancer Detection with InceptionV3

This notebook demonstrates the process of building a skin cancer detection model using the InceptionV3 architecture. The project involves:
1.  **Data Acquisition**: Downloading the HAM10000 dataset from Harvard Dataverse.
2.  **Data Preparation**: Organizing the dataset, labeling images as 'benign' or 'malignant', and splitting it into training, validation, and testing sets.
3.  **Model Development**: Utilizing a pre-trained InceptionV3 model from TensorFlow Keras, adding a custom classification head, and performing transfer learning.
4.  **Model Training**: Training the model in two phases: first, training only the custom head, and then fine-tuning the higher layers of the InceptionV3 base model.
5.  **Model Export**: Saving the trained model in Keras H5 format and converting it to TensorFlow Lite (TFLite) for potential deployment on mobile or edge devices.


## 1. Setting up the Environment and Downloading Data

This section handles mounting Google Drive, creating project directories, and downloading the necessary dataset (images and metadata) from Harvard Dataverse.

In [25]:
import os
import shutil

source_mobile_tflite = "skin_cancer_model_mobile.tflite"
dest_mobile_tflite = os.path.join(MODELS_DIR, source_mobile_tflite)

if os.path.exists(source_mobile_tflite):
    shutil.move(source_mobile_tflite, dest_mobile_tflite)
    print(f"Moved {source_mobile_tflite} to {MODELS_DIR}")
else:
    print(f"{source_mobile_tflite} not found in current directory.")

Moved skin_cancer_model_mobile.tflite to /content/drive/MyDrive/skin_cancer_project/models


In [26]:
import os

print(f"Contents of {MODELS_DIR} after moving mobile TFLite model:\n{os.listdir(MODELS_DIR)}")

Contents of /content/drive/MyDrive/skin_cancer_project/models after moving mobile TFLite model:
['skin_cancer_model.h5', 'skin_cancer_model.tflite', 'skin_cancer_model_mobile.tflite']


In [23]:
import tensorflow as tf

# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=os.path.join(MODELS_DIR, "skin_cancer_model.tflite"))
interpreter.allocate_tensors()

# Input details
print(interpreter.get_input_details())

# Output details
print(interpreter.get_output_details())


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


[{'name': 'serving_default_input_layer:0', 'index': 0, 'shape': array([  1, 299, 299,   3], dtype=int32), 'shape_signature': array([ -1, 299, 299,   3], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
[{'name': 'StatefulPartitionedCall_1:0', 'index': 319, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [24]:
import tensorflow as tf

# Load your saved model or h5 file
model = tf.keras.models.load_model('/content/drive/MyDrive/skin_cancer_project/models/skin_cancer_model.h5')  # or SavedModel path

# Convert WITHOUT select_tf_ops
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float32]

# DO NOT add this line - it causes the issue:
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]

# Only use builtin ops
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()

with open('skin_cancer_model_mobile.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model converted successfully!")

Saved artifact at '/tmp/tmpadbi2qmf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 299, 299, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  133315037903760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037901840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037904720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037902992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037902416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037901264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037903184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037901072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037901456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037902224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133315037901648